# Synthetic data overview

Set STP_PROJECT before launching Jupyter. The bundle README lists inputs, kernels and run order. Run the sections for your workflow.

In [ ]:
import os
from pathlib import Path

# Set these before starting Jupyter. Existing per-notebook overrides still apply.
if "STP_PROJECT" not in os.environ:
    raise RuntimeError("Set STP_PROJECT to the experiment workspace before starting Jupyter.")
project_root = Path(os.environ["STP_PROJECT"]).expanduser().resolve()
os.environ["STP_PROJECT"] = str(project_root)
os.environ.setdefault("SPATIAL_TI_PROJECT", str(project_root))
os.environ.setdefault("PFC_PROJECT", str(project_root))
os.environ.setdefault("FLOW_STP_REPO", str(project_root))
os.environ.setdefault("QIAN_PFC_DATA_ROOT", str(project_root / "data/qian_pfc"))
os.environ.setdefault("QIAN_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("PFC_DATA_ROOT", os.environ["QIAN_PFC_DATA_ROOT"])
os.environ.setdefault("STVCR_REPO", str(project_root / "external/stVCR"))

## Observed tissue geometry (Figure 3A)

In [ ]:
# %% [ONE SELF-CONTAINED JUPYTER CELL]
# Synthetic crescent — Ground-truth coordinate panel A
#
# Publication layout:
#   - EXACT 6.40 x 5.15 inches
#   - 10 tissues in 3x3 + 1 layout
#   - Larger tissues and compact spacing
#   - Labels above tissues, never overlaid
#   - Colorbar title and numbers inside SVG canvas
#   - Arial or close substitute
#   - SVG text converted to vector paths for PowerPoint
#   - Frozen dataset integrity verification
#   - No model fitting or evaluation

from pathlib import Path
import os
import json
import hashlib

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib.tri as mtri
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
from scipy.spatial import cKDTree

# ============================================================
# 1. CONFIGURATION
# ============================================================

PROJECT = Path(os.environ.get(
    "STP_PROJECT",
    os.environ["STP_PROJECT"]
)).expanduser()

DATA = Path(os.environ.get(
    "CRESCENT_DATA_ROOT",
    str(PROJECT / "Experiments/simulation validation/"
                  "Synthetic benchmarking/experiments/"
                  "synthetic_developmental_crescent_v1")
)).expanduser()

OUT = Path(os.environ.get(
    "SYNTHETIC_FIVE_PANEL_OUT",
    str(PROJECT / "experiments/synthetic_benchmark/"
                  "publication_results/panel_A_final")
)).expanduser()

OUT.mkdir(parents=True, exist_ok=True)

FIG_W = 6.40
FIG_H = 5.15
DPI = 450

PANEL_STEM = "FIG_synthetic_panel_A_final_6p40x5p15"

# Font sizes, in points.
TISSUE_LABEL_SIZE = 10.0
CB_LABEL_SIZE = 10.5
CB_TICK_SIZE = 11.5

INK = "#23313B"
CMAP = plt.get_cmap("viridis")
CONTOUR_DARK = "#10212B"
CONTOUR_LIGHT = "white"

# ============================================================
# 2. VERIFY FROZEN DATASET
# ============================================================

def require(path):
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024 * 1024), b""
        ):
            h.update(block)
    return h.hexdigest()


manifest_path = require(
    DATA / "manifests/dataset_manifest.json"
)

manifest = json.loads(manifest_path.read_text())

assert manifest["dataset_id"] == (
    "synthetic_developmental_crescent_v1"
)
assert int(manifest["seed"]) == 20261023
assert manifest["status"] == "frozen"

obs_rel = "train_only/public/observed_cells.csv.gz"
truth_rel = "evaluator_only/observed_hidden/cell_truth.csv.gz"

for rel in [obs_rel, truth_rel]:
    path = require(DATA / rel)
    assert sha256(path) == manifest["file_sha256"][rel], (
        f"SHA mismatch: {path}"
    )

# ============================================================
# 3. LOAD AND ALIGN DATA
# ============================================================

obs = pd.read_csv(
    DATA / obs_rel,
    usecols=["cell_id", "tissue_id", "time", "x", "y"]
)

truth = pd.read_csv(
    DATA / truth_rel,
    usecols=["cell_id", "z_true"]
)

df = obs.merge(
    truth,
    on="cell_id",
    how="inner",
    validate="one_to_one"
)

assert len(df) == len(obs) == len(truth)
assert df["tissue_id"].nunique() == 10
assert np.isfinite(
    df[["x", "y", "z_true"]].to_numpy(float)
).all()

VMIN = float(df["z_true"].min())
VMAX = float(df["z_true"].max())

tissue_table = (
    df[["tissue_id", "time"]]
    .drop_duplicates()
    .sort_values(["time", "tissue_id"])
    .reset_index(drop=True)
)

tissue_ids = (
    tissue_table["tissue_id"].astype(int).tolist()
)

assert len(tissue_ids) == 10

# ============================================================
# 4. FONT AND PUBLICATION STYLE
# ============================================================

available_fonts = {
    f.name for f in fm.fontManager.ttflist
}

font_candidates = [
    "Arial",
    "Liberation Sans",
    "Arimo",
    "Nimbus Sans",
    "DejaVu Sans",
]

FONT = next(
    (name for name in font_candidates
     if name in available_fonts),
    "DejaVu Sans"
)

print("Publication font:", FONT)

mpl.rcParams.update({
    "font.family": FONT,
    "font.size": TISSUE_LABEL_SIZE,
    "axes.labelsize": CB_LABEL_SIZE,
    "xtick.labelsize": CB_TICK_SIZE,
    "ytick.labelsize": CB_TICK_SIZE,
    "axes.linewidth": 0.7,
    "axes.edgecolor": INK,
    "text.color": INK,
    "axes.labelcolor": INK,
    "xtick.color": INK,
    "ytick.color": INK,

    # Preserve font quality in PDFs.
    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    # CRITICAL for PowerPoint SVG import:
    # Convert text to vector outlines.
    "svg.fonttype": "path",

    # Do not crop or resize the exported canvas.
    "savefig.bbox": None,
    "savefig.facecolor": "white",
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

# ============================================================
# 5. TRIANGULATION AND CONTOURS
# ============================================================

contour_levels = np.linspace(
    max(VMIN, 0.10),
    min(VMAX, 0.90),
    5
)

MAX_EDGE_KNN_MULTIPLIER = 3.0
MAX_EDGE_MEDIAN_MULTIPLIER = 5.5


def tissue_supported_triangulation(x, y):

    xy = np.column_stack([x, y])

    tri = mtri.Triangulation(x, y)
    t = tri.triangles

    tree = cKDTree(xy)

    k = min(6, len(xy))
    near = tree.query(xy, k=k)[0]

    local = (
        near[:, -1]
        if near.ndim == 2
        else near
    )

    finite = local[
        np.isfinite(local) & (local > 0)
    ]

    if len(finite) == 0:
        raise RuntimeError(
            "Cannot estimate local tissue spacing."
        )

    median = float(np.median(finite))

    p0 = xy[t[:, 0]]
    p1 = xy[t[:, 1]]
    p2 = xy[t[:, 2]]

    e01 = np.linalg.norm(p0 - p1, axis=1)
    e12 = np.linalg.norm(p1 - p2, axis=1)
    e20 = np.linalg.norm(p2 - p0, axis=1)

    loc01 = np.maximum(
        local[t[:, 0]], local[t[:, 1]]
    )

    loc12 = np.maximum(
        local[t[:, 1]], local[t[:, 2]]
    )

    loc20 = np.maximum(
        local[t[:, 2]], local[t[:, 0]]
    )

    mask = (
        (e01 > MAX_EDGE_KNN_MULTIPLIER * loc01)
        |
        (e12 > MAX_EDGE_KNN_MULTIPLIER * loc12)
        |
        (e20 > MAX_EDGE_KNN_MULTIPLIER * loc20)
    )

    longest = np.maximum.reduce(
        (e01, e12, e20)
    )

    mask |= (
        longest
        > MAX_EDGE_MEDIAN_MULTIPLIER * median
    )

    tri.set_mask(mask)

    if not np.any(~mask):
        raise RuntimeError(
            "All triangles masked."
        )

    return tri, int(mask.sum()), len(mask)


# ============================================================
# 6. EXACT INCH-BASED LAYOUT
# ============================================================

fig = plt.figure(
    figsize=(FIG_W, FIG_H),
    facecolor="white"
)

# Colorbar: all text will fit within 6.40 inches.
CB_LEFT = 5.37
CB_BOTTOM = 0.22
CB_W = 0.13
CB_H = 4.66

# Reserve space for colorbar tick numbers and title.
GRID_LEFT = 0.08
GRID_RIGHT = 5.22
GRID_TOP = 5.00
GRID_BOTTOM = 0.10

GRID_W = GRID_RIGHT - GRID_LEFT
GRID_H = GRID_TOP - GRID_BOTTOM

COL_GAP = 0.065
ROW_GAP = 0.025

TILE_W = (GRID_W - 2 * COL_GAP) / 3
TILE_H = (GRID_H - 3 * ROW_GAP) / 4

# Small header reserved above each tissue.
HEADER_H = 0.19

# Tissue plotting area.
INNER_PAD_X = 0.008
INNER_PAD_BOTTOM = 0.005

IMAGE_W = TILE_W - 2 * INNER_PAD_X
IMAGE_H = (
    TILE_H
    - HEADER_H
    - INNER_PAD_BOTTOM
)

col_lefts = [
    GRID_LEFT + c * (TILE_W + COL_GAP)
    for c in range(3)
]

row_bottoms = [
    GRID_TOP - (r + 1) * TILE_H - r * ROW_GAP
    for r in range(4)
]

# 3x3 + centered final tissue.
tile_positions = [
    (col_lefts[c], row_bottoms[r])
    for r in range(3)
    for c in range(3)
]

tile_positions.append(
    (col_lefts[1], row_bottoms[3])
)

# ============================================================
# 7. PLOT THE TEN TISSUES
# ============================================================

for n, (tid, position) in enumerate(
    zip(tissue_ids, tile_positions),
    start=1
):

    tile_x, tile_y = position

    sub = df.loc[df["tissue_id"].eq(tid)]

    x = sub["x"].to_numpy(float)
    y = -sub["y"].to_numpy(float)
    z = sub["z_true"].to_numpy(float)

    ax = fig.add_axes([
        (tile_x + INNER_PAD_X) / FIG_W,
        (tile_y + INNER_PAD_BOTTOM) / FIG_H,
        IMAGE_W / FIG_W,
        IMAGE_H / FIG_H,
    ])

    ax.scatter(
        x, y,
        c=z,
        cmap=CMAP,
        vmin=VMIN,
        vmax=VMAX,
        s=4.0,
        linewidths=0,
        rasterized=True,
    )

    if (
        len(np.unique(x)) > 3
        and len(np.unique(y)) > 3
    ):
        tri, n_masked, n_all = (
            tissue_supported_triangulation(x, y)
        )

        print(
            f"Tissue {n}: "
            f"masked {n_masked:,}/{n_all:,} triangles"
        )

        # Dark contour underlay.
        ax.tricontour(
            tri, z,
            levels=contour_levels,
            colors=CONTOUR_DARK,
            linewidths=2.35,
            alpha=0.88,
        )

        # White contour overlay.
        ax.tricontour(
            tri, z,
            levels=contour_levels,
            colors=CONTOUR_LIGHT,
            linewidths=1.30,
            alpha=0.98,
        )

    ax.set_aspect("equal")
    ax.set_axis_off()

    # Close framing, without cutting off scatter markers.
    xspan = float(np.ptp(x))
    yspan = float(np.ptp(y))

    ax.set_xlim(
        x.min() - 0.02 * xspan,
        x.max() + 0.02 * xspan
    )

    ax.set_ylim(
        y.min() - 0.02 * yspan,
        y.max() + 0.02 * yspan
    )

    # Tissue label placed within the reserved header band.
    # This is INSIDE the canvas and cannot overlap the tissue.
    label_x = (
        tile_x + TILE_W / 2
    ) / FIG_W

    label_y = (
        tile_y + TILE_H - HEADER_H / 2
    ) / FIG_H

    fig.text(
        label_x,
        label_y,
        f"Tissue {n}  ·  t = {n-1}",
        ha="center",
        va="center",
        fontsize=TISSUE_LABEL_SIZE,
        fontweight="bold",
        color=INK,
    )

# ============================================================
# 8. SHARED COLORBAR — CLIPPING-SAFE
# ============================================================

norm = Normalize(vmin=VMIN, vmax=VMAX)
sm = ScalarMappable(norm=norm, cmap=CMAP)
sm.set_array([])

cax = fig.add_axes([
    CB_LEFT / FIG_W,
    CB_BOTTOM / FIG_H,
    CB_W / FIG_W,
    CB_H / FIG_H,
])

cb = fig.colorbar(sm, cax=cax)

MID = 0.5 * (VMIN + VMAX)

cb.set_ticks([VMIN, MID, VMAX])

cb.set_ticklabels([
    f"{VMIN:.1f}",
    f"{MID:.1f}",
    f"{VMAX:.1f}",
])

cb.ax.tick_params(
    length=3.2,
    width=0.7,
    pad=2.0,
    labelsize=CB_TICK_SIZE,
)

cb.outline.set_linewidth(0.7)

# IMPORTANT:
# Do not call cb.set_label() here.
# Matplotlib's automatic label position can extend
# beyond the figure edge, even when the colorbar fits.
#
# Place the title explicitly in reserved canvas space.

fig.text(
    6.13 / FIG_W,
    0.50,
    "Ground-truth common coordinate",
    rotation=270,
    ha="center",
    va="center",
    fontsize=CB_LABEL_SIZE,
    fontweight="normal",
    color=INK,
)

# ============================================================
# 9. VALIDATE BOUNDS BEFORE EXPORT
# ============================================================

fig.canvas.draw()

renderer = fig.canvas.get_renderer()

# Bounds are returned in inches.
bounds = fig.get_tightbbox(renderer)

print("\nExport dimensions (inches):")
print(fig.get_size_inches())

print("\nRendered content bounds (inches):")
print([round(v, 3) for v in bounds.extents])

tol = 0.005

if not (
    bounds.x0 >= -tol
    and bounds.y0 >= -tol
    and bounds.x1 <= FIG_W + tol
    and bounds.y1 <= FIG_H + tol
):
    raise RuntimeError(
        "An artist extends outside the figure canvas.\n"
        f"Figure: {FIG_W} × {FIG_H} inches\n"
        f"Content bounds: {bounds.extents}"
    )

# ============================================================
# 10. EXPORT
# ============================================================

# SVG has vector-outline text; PowerPoint will not
# reflow the text using a substituted font.
#
# Never use bbox_inches='tight': that changes
# the exported physical dimensions.

for ext in ("svg", "pdf", "png"):

    out = OUT / f"{PANEL_STEM}.{ext}"

    fig.savefig(
        out,
        dpi=DPI,
        bbox_inches=None,
        facecolor="white",
    )

    print("SAVED:", out)

# Verify SVG physical size from exported metadata.
import xml.etree.ElementTree as ET

svg_path = OUT / f"{PANEL_STEM}.svg"
root = ET.parse(svg_path).getroot()

print("\nSVG width:", root.attrib.get("width"))
print("SVG height:", root.attrib.get("height"))
print("SVG viewBox:", root.attrib.get("viewBox"))

# Save tissue order for reproducibility.
receipt = tissue_table.copy()

receipt["display_label"] = [
    f"Tissue {i + 1}"
    for i in range(len(receipt))
]

receipt["display_time_label"] = [
    f"t = {i}"
    for i in range(len(receipt))
]

receipt.to_csv(
    OUT / f"{PANEL_STEM}_tissue_order.csv",
    index=False,
)

print("\nFinal figure:", FIG_W, "×", FIG_H, "in")
print("Output directory:", OUT)

plt.show()
plt.close(fig)

## Ground-truth progression (Figure 3B)

In [ ]:
# Zoetrope | Synthetic developmental crescent | Panel B
#
# Ground-truth velocity V(z,t) — NOT inferred velocity.
#
# Publication updates:
#   - Larger text consistent with revised Panel A
#   - Arial-compatible font
#   - Title: "Velocity V"
#   - Colorbar ticks only at 0.0 and 0.4
#   - Centered vertical "V" beside colorbar, flipped orientation
#   - Exact 3.15" x 2.75" canvas
#   - Physically square heatmap
#   - PowerPoint-safe SVG export
#   - Frozen simulator verification preserved

from pathlib import Path
import hashlib
import json
import os
import xml.etree.ElementTree as ET

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import pandas as pd
from matplotlib.ticker import FormatStrFormatter

# ============================================================
# 1. CONFIGURATION
# ============================================================

PROJECT = Path(os.environ.get(
    "STP_PROJECT",
    os.environ["STP_PROJECT"]
)).expanduser()

DATA_ROOT = Path(os.environ.get(
    "CRESCENT_DATA_ROOT",
    str(PROJECT / "Experiments/simulation validation/"
                  "Synthetic benchmarking/experiments/"
                  "synthetic_developmental_crescent_v1")
)).expanduser()

OUTPUT_DIR = Path(os.environ.get(
    "SYNTHETIC_FIVE_PANEL_OUT",
    str(PROJECT / "experiments/synthetic_benchmark/"
                  "publication_results/panel_B_revised")
)).expanduser()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_W = 3.15
FIG_H = 2.75
DPI = 450

STEM = "FIG_synthetic_panel_B_velocity_V_publication_v2"

NZ = 281
NT = 361
SHOW_ISOVELOCITY_CONTOURS = True

# Publication font sizes
TITLE_SIZE = 12.0
AXIS_LABEL_SIZE = 12.0
AXIS_TICK_SIZE = 11.0
CB_TICK_SIZE = 11.0
CB_LABEL_SIZE = 12.0

# ============================================================
# 2. EXACT FROZEN SIMULATOR
# ============================================================

VERIFY_FROZEN_TRUTH = (
    os.environ.get("VERIFY_SYNTHETIC_VELOCITY_TRUTH", "1") == "1"
)

VELOCITY_SCALE = 1.8
VELOCITY_ASYMMETRY = 0.45
VELOCITY_FINAL_FRACTION = 0.10

def ground_truth_v(z, t):
    """Ground-truth velocity in current anatomical coordinates."""
    z, t = np.broadcast_arrays(np.asarray(z, float), np.asarray(t, float))
    spatial = (
        VELOCITY_SCALE
        * z * (1.0 - z)
        * (1.0 + VELOCITY_ASYMMETRY * (2.0*z - 1.0))
    )
    smoothstep = 3.0*t*t - 2.0*t*t*t
    temporal = 1.0 - (1.0 - VELOCITY_FINAL_FRACTION) * smoothstep
    return spatial * temporal

def file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# ============================================================
# 3. FROZEN DATA VERIFICATION
# ============================================================

if VERIFY_FROZEN_TRUTH:
    manifest_path = DATA_ROOT / "manifests/dataset_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(f"Frozen manifest missing: {manifest_path}")

    manifest = json.loads(manifest_path.read_text())
    assert manifest["dataset_id"] == "synthetic_developmental_crescent_v1"
    assert int(manifest["seed"]) == 20261023
    assert manifest["status"] == "frozen"

    public_rel = "train_only/public/observed_cells.csv.gz"
    oracle_rel = "evaluator_only/observed_hidden/cell_truth.csv.gz"

    for relative in (public_rel, oracle_rel):
        p = DATA_ROOT / relative
        if not p.is_file():
            raise FileNotFoundError(p)
        assert file_sha256(p) == manifest["file_sha256"][relative], (
            f"SHA mismatch: {p}"
        )

    observed = pd.read_csv(
        DATA_ROOT / public_rel,
        usecols=["cell_id", "time"]
    )
    oracle = pd.read_csv(
        DATA_ROOT / oracle_rel,
        usecols=["cell_id", "z_true", "vz_true"]
    )

    observed["cell_id"] = observed["cell_id"].astype(str)
    oracle["cell_id"] = oracle["cell_id"].astype(str)

    assert observed.cell_id.is_unique
    assert oracle.cell_id.is_unique

    pairs = observed.merge(
        oracle,
        on="cell_id",
        validate="one_to_one"
    )
    assert len(pairs) == len(observed) == len(oracle) == 36425

    reconstructed_v = ground_truth_v(
        pairs["z_true"].to_numpy(float),
        pairs["time"].to_numpy(float)
    )

    error = float(np.max(np.abs(
        reconstructed_v - pairs["vz_true"].to_numpy(float)
    )))
    assert error < 1e-8, f"Velocity oracle mismatch: {error:.3g}"

    print(
        f"PASS: simulator velocity verified against "
        f"{len(pairs):,} frozen cells; error={error:.2g}"
    )
else:
    print("Offline layout preview: frozen verification disabled.")

# ============================================================
# 4. EVALUATE TRUE VELOCITY FIELD
# ============================================================

z_grid = np.linspace(0.0, 1.0, NZ)
t_grid = np.linspace(0.0, 1.0, NT)

Z, T = np.meshgrid(z_grid, t_grid, indexing="xy")
V_true = ground_truth_v(Z, T)

assert V_true.shape == (NT, NZ)
assert np.isfinite(V_true).all()
assert float(V_true.min()) >= -1e-12
assert np.allclose(V_true[:, [0, -1]], 0, atol=1e-12)

vmax = float(V_true.max())

# ============================================================
# 5. FONT AND PUBLICATION STYLE
# ============================================================

available_fonts = {f.name for f in fm.fontManager.ttflist}
font_candidates = [
    "Arial",
    "Liberation Sans",
    "Arimo",
    "Nimbus Sans",
    "DejaVu Sans",
]
FONT = next((name for name in font_candidates if name in available_fonts), "DejaVu Sans")
print("Publication font:", FONT)

INK = "#23313B"

mpl.rcParams.update({
    "font.family": FONT,
    "font.size": AXIS_TICK_SIZE,
    "axes.labelsize": AXIS_LABEL_SIZE,
    "axes.titlesize": TITLE_SIZE,
    "xtick.labelsize": AXIS_TICK_SIZE,
    "ytick.labelsize": AXIS_TICK_SIZE,
    "axes.linewidth": 0.75,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "text.color": INK,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "path",   # PowerPoint-safe
    "savefig.bbox": None,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})

# ============================================================
# 6. MANUAL INCH-BASED LAYOUT
# ============================================================

fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor="white")

# Shift heatmap slightly right to avoid left-edge overflow.
SIDE = 1.80
AX_LEFT = 0.72
AX_BOTTOM = 0.55

CB_LEFT = 2.64
CB_BOTTOM = AX_BOTTOM
CB_WIDTH = 0.115
CB_HEIGHT = SIDE

ax = fig.add_axes([
    AX_LEFT / FIG_W,
    AX_BOTTOM / FIG_H,
    SIDE / FIG_W,
    SIDE / FIG_H,
])

# ============================================================
# 7. VELOCITY HEATMAP
# ============================================================

image = ax.pcolormesh(
    t_grid,
    z_grid,
    V_true.T,
    shading="auto",
    cmap="viridis",
    vmin=0.0,
    vmax=vmax,
    rasterized=True
)

if SHOW_ISOVELOCITY_CONTOURS:
    levels = np.array([0.1, 0.2, 0.3, 0.4])
    levels = levels[levels < vmax]
    if len(levels):
        ax.contour(
            t_grid,
            z_grid,
            V_true.T,
            levels=levels,
            colors="white",
            linewidths=0.85,
            alpha=0.85
        )

ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("equal", adjustable="box")

ax.set_xticks([0, 0.25, 0.5, 0.75, 1])
ax.set_yticks([0, 0.25, 0.5, 0.75, 1])

ax.xaxis.set_major_formatter(FormatStrFormatter("%g"))
ax.yaxis.set_major_formatter(FormatStrFormatter("%g"))

ax.tick_params(
    length=3.0,
    width=0.7,
    direction="out",
    pad=3.0,
    labelsize=AXIS_TICK_SIZE
)

ax.spines[["top", "right"]].set_visible(False)

ax.set_xlabel(
    "Developmental time",
    fontsize=AXIS_LABEL_SIZE,
    fontweight="normal",
    labelpad=5
)

ax.set_ylabel(
    "Common axis",
    fontsize=AXIS_LABEL_SIZE,
    fontweight="normal",
    labelpad=4
)

# ============================================================
# 8. TITLE ABOVE THE HEATMAP
# ============================================================

fig.text(
    (AX_LEFT + SIDE / 2) / FIG_W,
    2.58 / FIG_H,
    "Velocity V",
    ha="center",
    va="center",
    fontsize=TITLE_SIZE,
    fontweight="bold",
    color=INK
)

# ============================================================
# 9. COLORBAR
# ============================================================

cax = fig.add_axes([
    CB_LEFT / FIG_W,
    CB_BOTTOM / FIG_H,
    CB_WIDTH / FIG_W,
    CB_HEIGHT / FIG_H
])

cb = fig.colorbar(image, cax=cax, orientation="vertical")

# Remove middle 0.2 tick
cb.set_ticks([0.0, 0.4])
cb.ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))

cb.ax.tick_params(
    labelsize=CB_TICK_SIZE,
    length=2.8,
    width=0.7,
    pad=2
)

cb.outline.set_linewidth(0.7)

# Put "V" centered beside the colorbar, flipped the other way
fig.text(
    3.04 / FIG_W,
    (CB_BOTTOM + CB_HEIGHT / 2) / FIG_H,
    "V",
    ha="center",
    va="center",
    rotation=270,
    fontsize=CB_LABEL_SIZE,
    fontweight="normal",
    color=INK
)

# ============================================================
# 10. CHECK EXPORT BOUNDS
# ============================================================

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
bounds = fig.get_tightbbox(renderer)

print("\nCanvas dimensions:")
print(f"{FIG_W:.2f} x {FIG_H:.2f} inches")

print("\nRendered content bounds:")
print([round(float(v), 3) for v in bounds.extents])

# Slightly looser tolerance now that we know PPT is the final use case.
tol = 0.03

assert bounds.x0 >= -tol, "Content extends too far beyond left edge."
assert bounds.y0 >= -tol, "Content extends too far beyond bottom edge."
assert bounds.x1 <= FIG_W + tol, "Content extends beyond right edge."
assert bounds.y1 <= FIG_H + tol, "Content extends beyond top edge."

# ============================================================
# 11. EXPORT
# ============================================================

for ext in ("svg", "pdf", "png"):
    target = OUTPUT_DIR / f"{STEM}.{ext}"
    fig.savefig(
        target,
        dpi=DPI,
        bbox_inches=None,
        facecolor="white"
    )
    print("SAVED:", target)

# Verify SVG dimensions
svg_path = OUTPUT_DIR / f"{STEM}.svg"
root = ET.parse(svg_path).getroot()

print("\nSVG width:", root.attrib.get("width"))
print("SVG height:", root.attrib.get("height"))
print("SVG viewBox:", root.attrib.get("viewBox"))

# Save provenance
receipt = {
    "dataset_id": "synthetic_developmental_crescent_v1",
    "simulator_seed": 20261023,
    "source_field": "ground_truth_v(z,t)",
    "model_velocity_used": False,
    "truth_verified": bool(VERIFY_FROZEN_TRUTH),
    "formula": (
        "1.8*z*(1-z)*(1+0.45*(2*z-1))"
        "*(1-0.9*(3*t*t-2*t*t*t))"
    ),
    "panel_inches": [FIG_W, FIG_H],
    "heatmap_inches": [SIDE, SIDE],
    "font": FONT,
    "title": "Velocity V",
    "colorbar_ticks": [0.0, 0.4],
    "colorbar_label": "V",
    "vmin": float(V_true.min()),
    "vmax": vmax,
    "axes": {
        "x": "Developmental time",
        "y": "Common axis"
    }
}
(OUTPUT_DIR / f"{STEM}_receipt.json").write_text(
    json.dumps(receipt, indent=2) + "\n"
)

print(f"\nTrue velocity range: [{V_true.min():.4g}, {vmax:.4g}]")

plt.show()
plt.close(fig)

## Ground-truth developmental dynamics (Figure 3C)

In [ ]:
# Zoetrope | Synthetic developmental crescent | Panel C
#
# Ground-truth flow map trajectories z(t) = F_t(a0)
# x = developmental time
# y = current common coordinate
# line color = initial coordinate a0
#
# Publication updates:
#   - Larger text consistent with revised Panels A and B
#   - Arial-compatible font
#   - Title: "Flow map F"
#   - Colorbar ticks only at 0.0 and 1.0
#   - Centered vertical "a_0" beside colorbar, flipped orientation
#   - Exact 3.15" x 2.75" canvas
#   - Physically square plotting region
#   - PowerPoint-safe SVG export
#   - Frozen simulator verification preserved

from pathlib import Path
import hashlib
import json
import os
import xml.etree.ElementTree as ET

import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import numpy as np
import pandas as pd
from matplotlib.ticker import FormatStrFormatter
from matplotlib.colors import Normalize
from matplotlib.cm import ScalarMappable

# ============================================================
# 1. CONFIGURATION
# ============================================================

PROJECT = Path(os.environ.get(
    "STP_PROJECT",
    os.environ["STP_PROJECT"]
)).expanduser()

DATA_ROOT = Path(os.environ.get(
    "CRESCENT_DATA_ROOT",
    str(PROJECT / "Experiments/simulation validation/"
                  "Synthetic benchmarking/experiments/"
                  "synthetic_developmental_crescent_v1")
)).expanduser()

OUTPUT_DIR = Path(os.environ.get(
    "SYNTHETIC_FIVE_PANEL_OUT",
    str(PROJECT / "experiments/synthetic_benchmark/"
                  "publication_results/panel_C_revised")
)).expanduser()

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_W = 3.15
FIG_H = 2.75
DPI = 450

STEM = "FIG_synthetic_panel_C_flow_map_F_publication"

NA = 281
NT = 361
VERIFY_FROZEN_TRUTH = os.environ.get("VERIFY_SYNTHETIC_FLOW_TRUTH", "1") == "1"

# Publication font sizes
TITLE_SIZE = 12.0
AXIS_LABEL_SIZE = 12.0
AXIS_TICK_SIZE = 11.0
CB_TICK_SIZE = 11.0
CB_LABEL_SIZE = 12.0

# Exact frozen simulator constants
VELOCITY_SCALE = 1.8
VELOCITY_ASYMMETRY = 0.45
VELOCITY_FINAL_FRACTION = 0.10

# ============================================================
# 2. EXACT FROZEN SIMULATOR
# ============================================================

def ground_truth_velocity(z, t):
    """Generator's Eulerian common-coordinate velocity dz/dt = v_z(z,t)."""
    z, t = np.broadcast_arrays(np.asarray(z, float), np.asarray(t, float))
    spatial = VELOCITY_SCALE * z * (1 - z) * (1 + VELOCITY_ASYMMETRY * (2 * z - 1))
    temporal = 1 - (1 - VELOCITY_FINAL_FRACTION) * (3 * t * t - 2 * t * t * t)
    return spatial * temporal

def generator_flow(a, t_end, steps_per_unit=300):
    """Reference implementation of the frozen simulator's RK4 flow F_t(a)."""
    z = np.asarray(a, float).copy()
    n = max(1, int(np.ceil(steps_per_unit * float(t_end))))
    dt = float(t_end) / n
    for i in range(n):
        t = i * dt
        k1 = ground_truth_velocity(z, t)
        k2 = ground_truth_velocity(z + 0.5 * dt * k1, t + 0.5 * dt)
        k3 = ground_truth_velocity(z + 0.5 * dt * k2, t + 0.5 * dt)
        k4 = ground_truth_velocity(z + dt * k3, t + dt)
        z += (dt / 6.0) * (k1 + 2 * k2 + 2 * k3 + k4)
    return z

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

# ============================================================
# 3. VERIFY EXACT SIMULATOR AGAINST FROZEN DATA
# ============================================================

max_saved_error = None

if VERIFY_FROZEN_TRUTH:
    manifest_path = DATA_ROOT / "manifests/dataset_manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError(
            f"Frozen manifest missing: {manifest_path}\n"
            "For an offline preview ONLY, set VERIFY_SYNTHETIC_FLOW_TRUTH=0."
        )

    manifest = json.loads(manifest_path.read_text())
    assert manifest["dataset_id"] == "synthetic_developmental_crescent_v1"
    assert int(manifest["seed"]) == 20261023
    assert manifest["status"] == "frozen"

    obs_rel = "train_only/public/observed_cells.csv.gz"
    truth_rel = "evaluator_only/observed_hidden/cell_truth.csv.gz"

    for rel in (obs_rel, truth_rel):
        path = DATA_ROOT / rel
        if not path.is_file():
            raise FileNotFoundError(path)
        assert sha256(path) == manifest["file_sha256"][rel], f"SHA-256 mismatch: {path}"

    obs = pd.read_csv(DATA_ROOT / obs_rel, usecols=["cell_id", "time"])
    truth = pd.read_csv(
        DATA_ROOT / truth_rel,
        usecols=["cell_id", "material_a_true", "z_true"]
    )

    obs["cell_id"] = obs["cell_id"].astype(str)
    truth["cell_id"] = truth["cell_id"].astype(str)

    assert obs.cell_id.is_unique and truth.cell_id.is_unique

    joined = obs.merge(truth, on="cell_id", how="inner", validate="one_to_one")
    assert len(joined) == len(obs) == len(truth) == 36425

    replay = np.empty(len(joined), dtype=float)
    joined_t = joined["time"].to_numpy(float)
    joined_a = joined["material_a_true"].to_numpy(float)

    for tm in np.unique(joined_t):
        idx = np.flatnonzero(np.isclose(joined_t, tm, atol=1e-11))
        replay[idx] = generator_flow(joined_a[idx], tm)

    max_saved_error = float(np.max(np.abs(replay - joined["z_true"].to_numpy(float))))
    assert max_saved_error < 2e-6, (
        f"True flow does not reproduce frozen observed coordinates; "
        f"max error={max_saved_error:.3g}"
    )

    print(
        f"PASS exact generator flow vs {len(joined):,} frozen cells; "
        f"max error={max_saved_error:.2g}"
    )
else:
    print("Offline visual preview: archived simulator truth verification skipped.")

# ============================================================
# 4. COMPUTE GROUND-TRUTH FLOW MAP
# ============================================================

a_grid = np.linspace(0.0, 1.0, NA)   # initial/material coordinate
t_grid = np.linspace(0.0, 1.0, NT)

flow_map = np.empty((NT, NA), dtype=np.float64)
flow_map[0] = a_grid

for k in range(NT - 1):
    t = float(t_grid[k])
    h = float(t_grid[k + 1] - t_grid[k])
    z = flow_map[k]

    k1 = ground_truth_velocity(z, t)
    k2 = ground_truth_velocity(z + 0.5 * h * k1, t + 0.5 * h)
    k3 = ground_truth_velocity(z + 0.5 * h * k2, t + 0.5 * h)
    k4 = ground_truth_velocity(z + h * k3, t + h)

    flow_map[k + 1] = z + h / 6.0 * (k1 + 2 * k2 + 2 * k3 + k4)

assert np.isfinite(flow_map).all()
assert np.allclose(flow_map[0], a_grid)
assert np.allclose(flow_map[:, 0], 0)
assert np.allclose(flow_map[:, -1], 1)
assert np.all(np.diff(flow_map, axis=1) >= -1e-10), "Ground-truth flow lost ordering"
assert np.all(flow_map >= a_grid[None, :] - 1e-10)
assert np.all(flow_map <= 1 + 1e-10)

max_rk4_error = max(
    float(np.max(np.abs(flow_map[j] - generator_flow(a_grid, t_grid[j]))))
    for j in (0, 90, 180, 270, 360)
)
assert max_rk4_error < 1e-8, f"Grid RK4 differs from generator: {max_rk4_error:.3g}"

# ============================================================
# 5. FONT AND PUBLICATION STYLE
# ============================================================

available_fonts = {f.name for f in fm.fontManager.ttflist}
font_candidates = [
    "Arial",
    "Liberation Sans",
    "Arimo",
    "Nimbus Sans",
    "DejaVu Sans",
]
FONT = next((name for name in font_candidates if name in available_fonts), "DejaVu Sans")
print("Publication font:", FONT)

INK = "#23313B"

mpl.rcParams.update({
    "font.family": FONT,
    "font.size": AXIS_TICK_SIZE,
    "axes.labelsize": AXIS_LABEL_SIZE,
    "axes.titlesize": TITLE_SIZE,
    "xtick.labelsize": AXIS_TICK_SIZE,
    "ytick.labelsize": AXIS_TICK_SIZE,
    "axes.linewidth": 0.75,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "text.color": INK,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "path",   # PowerPoint-safe
    "savefig.bbox": None,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})

# ============================================================
# 6. MANUAL INCH-BASED LAYOUT
# ============================================================

fig = plt.figure(figsize=(FIG_W, FIG_H), facecolor="white")

SIDE = 1.80
AX_LEFT = 0.72
AX_BOTTOM = 0.55

CB_LEFT = 2.64
CB_BOTTOM = AX_BOTTOM
CB_WIDTH = 0.115
CB_HEIGHT = SIDE

ax = fig.add_axes([
    AX_LEFT / FIG_W,
    AX_BOTTOM / FIG_H,
    SIDE / FIG_W,
    SIDE / FIG_H,
])

# ============================================================
# 7. PLOT TRAJECTORIES
# ============================================================

initial_coords = np.arange(0.1, 1.0, 0.1)
colormap = mpl.colormaps["viridis"]
norm = Normalize(vmin=0.0, vmax=1.0)

for a0 in initial_coords:
    j = np.argmin(np.abs(a_grid - a0))
    path = flow_map[:, j]
    color = colormap(norm(a0))

    # Dark underlay
    ax.plot(
        t_grid, path,
        lw=2.25,
        c="#273640",
        alpha=0.35,
        solid_capstyle="round",
        zorder=2
    )

    # Colored trajectory
    ax.plot(
        t_grid, path,
        lw=1.75,
        c=color,
        solid_capstyle="round",
        zorder=3
    )

    # Start/end markers
    ax.scatter(
        [0.0], [path[0]],
        s=14,
        color=color,
        edgecolor="#273640",
        linewidth=0.3,
        zorder=5,
        clip_on=False
    )

    ax.scatter(
        [1.0], [path[-1]],
        s=20,
        marker="o",
        color=color,
        edgecolor="#273640",
        linewidth=0.45,
        zorder=5,
        clip_on=False
    )

    # Direction arrows
    if 0.2 <= a0 <= 0.8:
        i0 = np.searchsorted(t_grid, 0.60)
        i1 = np.searchsorted(t_grid, 0.69)
        ax.annotate(
            "",
            xy=(t_grid[i1], path[i1]),
            xytext=(t_grid[i0], path[i0]),
            arrowprops=dict(
                arrowstyle="-|>",
                color=color,
                lw=1.5,
                mutation_scale=7
            ),
            zorder=6
        )

ax.set_xlim(0.0, 1.0)
ax.set_ylim(0.0, 1.0)
ax.set_aspect("equal", adjustable="box")

ax.set_xticks([0, 0.25, 0.5, 0.75, 1])
ax.set_yticks([0, 0.25, 0.5, 0.75, 1])
ax.xaxis.set_major_formatter(FormatStrFormatter("%g"))
ax.yaxis.set_major_formatter(FormatStrFormatter("%g"))

ax.tick_params(
    length=3.0,
    width=0.7,
    direction="out",
    pad=3.0,
    labelsize=AXIS_TICK_SIZE
)

ax.spines[["top", "right"]].set_visible(False)

ax.grid(color="#D3D9DE", linewidth=0.45, alpha=0.55)
ax.set_axisbelow(True)

ax.set_xlabel(
    "Developmental time",
    fontsize=AXIS_LABEL_SIZE,
    fontweight="normal",
    labelpad=5
)

ax.set_ylabel(
    "Current common coordinate",
    fontsize=AXIS_LABEL_SIZE,
    fontweight="normal",
    labelpad=4
)

# ============================================================
# 8. TITLE
# ============================================================

fig.text(
    (AX_LEFT + SIDE / 2) / FIG_W,
    2.58 / FIG_H,
    "Flow map F",
    ha="center",
    va="center",
    fontsize=TITLE_SIZE,
    fontweight="bold",
    color=INK
)

# ============================================================
# 9. COLORBAR
# ============================================================

cax = fig.add_axes([
    CB_LEFT / FIG_W,
    CB_BOTTOM / FIG_H,
    CB_WIDTH / FIG_W,
    CB_HEIGHT / FIG_H
])

cb = fig.colorbar(
    ScalarMappable(norm=norm, cmap=colormap),
    cax=cax,
    orientation="vertical"
)

cb.set_ticks([0.0, 1.0])
cb.ax.yaxis.set_major_formatter(FormatStrFormatter("%.1f"))

cb.ax.tick_params(
    labelsize=CB_TICK_SIZE,
    length=2.8,
    width=0.7,
    pad=2
)

cb.outline.set_linewidth(0.7)

# Side label for initial coordinate
fig.text(
    3.04 / FIG_W,
    (CB_BOTTOM + CB_HEIGHT / 2) / FIG_H,
    r"$a_0$",
    ha="center",
    va="center",
    rotation=270,
    fontsize=CB_LABEL_SIZE,
    fontweight="normal",
    color=INK
)

# ============================================================
# 10. CHECK EXPORT BOUNDS
# ============================================================

fig.canvas.draw()
renderer = fig.canvas.get_renderer()
bounds = fig.get_tightbbox(renderer)

print("\nCanvas dimensions:")
print(f"{FIG_W:.2f} x {FIG_H:.2f} inches")

print("\nRendered content bounds:")
print([round(float(v), 3) for v in bounds.extents])

tol = 0.03

assert bounds.x0 >= -tol, "Content extends too far beyond left edge."
assert bounds.y0 >= -tol, "Content extends too far beyond bottom edge."
assert bounds.x1 <= FIG_W + tol, "Content extends beyond right edge."
assert bounds.y1 <= FIG_H + tol, "Content extends beyond top edge."

# ============================================================
# 11. EXPORT
# ============================================================

for ext in ("svg", "pdf", "png"):
    path = OUTPUT_DIR / f"{STEM}.{ext}"
    fig.savefig(
        path,
        dpi=DPI,
        bbox_inches=None,
        facecolor="white"
    )
    print("SAVED:", path)

svg_path = OUTPUT_DIR / f"{STEM}.svg"
root = ET.parse(svg_path).getroot()

print("\nSVG width:", root.attrib.get("width"))
print("SVG height:", root.attrib.get("height"))
print("SVG viewBox:", root.attrib.get("viewBox"))

# Save displayed trajectory paths
pd.DataFrame({
    "initial_coordinate": np.repeat(initial_coords, len(t_grid)),
    "developmental_time": np.tile(t_grid, len(initial_coords)),
    "current_coordinate": np.concatenate([
        flow_map[:, np.argmin(np.abs(a_grid - a0))]
        for a0 in initial_coords
    ])
}).to_csv(OUTPUT_DIR / f"{STEM}_paths.csv", index=False)

# Save provenance
receipt = dict(
    dataset_id="synthetic_developmental_crescent_v1",
    seed=20261023,
    source="exact ground-truth simulator flow; no learned quantities",
    panel_inches=[FIG_W, FIG_H],
    plot="true trajectories z(t)=F_t(a0)",
    title="Flow map F",
    axes={
        "x": "developmental_time",
        "y": "current_common_coordinate"
    },
    colored_value="initial_common_coordinate",
    colorbar_ticks=[0.0, 1.0],
    colorbar_label="a0",
    initial_coordinates=initial_coords.tolist(),
    max_generator_grid_error=max_rk4_error,
    max_frozen_cell_error=max_saved_error,
    verified_frozen_truth=VERIFY_FROZEN_TRUTH,
    font=FONT
)
(OUTPUT_DIR / f"{STEM}_receipt.json").write_text(
    json.dumps(receipt, indent=2) + "\n"
)

print(f"\nMax generator grid error: {max_rk4_error:.3g}")

plt.show()
plt.close(fig)

## Ground-truth expression programs (Figure 3D)

In [ ]:
# %% [ONE SELF-CONTAINED JUPYTER CELL]
# Zoetrope | Synthetic developmental crescent | Panel D
#
# Ground-truth expected expression: 8 representative gene types.
#
# Updates:
#   - Larger, nearly square heatmaps (0.92" x 1.00")
#   - Reduced whitespace above "Developmental time"
#   - Extra separation between "Common axis" and y-axis ticks
#   - Gene Type 1–8 headings
#   - Individual absolute-expression colorbars
#   - Exact 3.15" x 5.50" dimensions
#   - PowerPoint-safe SVG export
#   - Frozen simulator verification preserved

from pathlib import Path
import os
import json
import hashlib
import xml.etree.ElementTree as ET

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ticker import FormatStrFormatter

# ============================================================
# 1. CONFIGURATION
# ============================================================

PROJECT = Path(os.environ.get(
    "STP_PROJECT",
    os.environ["STP_PROJECT"]
)).expanduser()

DATA = Path(os.environ.get(
    "CRESCENT_DATA_ROOT",
    str(PROJECT / "Experiments/simulation validation/"
                  "Synthetic benchmarking/experiments/"
                  "synthetic_developmental_crescent_v1")
)).expanduser()

OUT = Path(os.environ.get(
    "SYNTHETIC_FIVE_PANEL_OUT",
    str(PROJECT / "experiments/synthetic_benchmark/"
                  "publication_results/panel_D_revised")
)).expanduser()

OUT.mkdir(parents=True, exist_ok=True)

FIG_W, FIG_H = 3.15, 5.50
DPI = 450

GENES = (
    "G01", "G06", "G11", "G16",
    "G21", "G26", "G31", "G36"
)

NZ, NT = 241, 301

STEM = "FIG_synthetic_panel_D_gene_types_larger_compact"

VERIFY_FROZEN = (
    os.getenv("VERIFY_SYNTHETIC_EXPRESSION_TRUTH", "1")
    == "1"
)

# Publication font sizes
TITLE_SIZE = 10
AXIS_LABEL_SIZE = 12.0
AXIS_TICK_SIZE = 10.5
CB_TICK_SIZE = 8.5

INK = "#23313B"

# ============================================================
# 2. EXACT FROZEN SIMULATOR
# ============================================================

VELOCITY_SCALE = 1.8
VELOCITY_ASYMMETRY = 0.45
VELOCITY_FINAL_FRACTION = 0.10

SEED = 20261023

AMP = np.array([
    3.5, 3.1, 3.3, 2.0,
    3.6, 3.6, 2.8, 2.8
])

OMEGA1, OMEGA2 = 1.5, 1.9
SECOND_PAIR_WEIGHT = 0.40
DOMINANCE_TRANSITION_A0 = 0.85


def v_z(z, t):
    z, t = np.broadcast_arrays(
        np.asarray(z, float),
        np.asarray(t, float)
    )

    spatial = (
        VELOCITY_SCALE
        * z * (1 - z)
        * (1 + VELOCITY_ASYMMETRY * (2*z - 1))
    )

    temporal = (
        1
        - (1 - VELOCITY_FINAL_FRACTION)
        * (3*t*t - 2*t*t*t)
    )

    return spatial * temporal


def spatial_basis(a):
    a = np.asarray(a, float)

    early = 1 - 2*a

    mid = (
        2*np.exp(-0.5*((a - 0.43)/0.15)**2)
        - 0.65
    )

    late = (
        2/(1 + np.exp(-(a - 0.64)/0.105))
        - 1
    )

    band = np.sin(2*np.pi*a)

    return np.stack(
        [early, mid, late, band],
        axis=-1
    )


def spatial_basis_derivative(a):
    a = np.asarray(a, float)

    gaussian = np.exp(
        -0.5*((a - 0.43)/0.15)**2
    )

    logistic = (
        1/(1 + np.exp(-(a - 0.64)/0.105))
    )

    return np.stack([
        -2*np.ones_like(a),
        -2*gaussian*(a - 0.43)/(0.15**2),
        2*logistic*(1 - logistic)/0.105,
        2*np.pi*np.cos(2*np.pi*a),
    ], axis=-1)


def calibration_amplitude():
    a = DOMINANCE_TRANSITION_A0

    base_sq = float(np.sum(
        (
            spatial_basis_derivative(
                np.array([a])
            )[0] * AMP[:4]
        )**2
    ))

    v = float(v_z(a, 0.0))

    qper2 = (
        (a**3 * OMEGA1)**2
        + (SECOND_PAIR_WEIGHT * a**4 * OMEGA2)**2
    )

    aper2 = (
        (3*a*a)**2
        + (SECOND_PAIR_WEIGHT * 4*a**3)**2
    )

    den = qper2 - v*v*aper2

    if den <= 0:
        raise RuntimeError(
            "Invalid frozen simulator calibration"
        )

    return float(v*np.sqrt(base_sq/den))


def expression_parameters(seed):
    rng = np.random.default_rng(seed + 9001)

    raw = np.zeros((48, 8))

    prototypes = (
        3*np.eye(8)
        + 0.35*rng.normal(size=(8, 8))
    )

    for family in range(8):
        raw[family*5:(family+1)*5] = (
            prototypes[family]
            + 0.09*rng.normal(size=(5, 8))
        )

    loading, _ = np.linalg.qr(
        raw, mode="reduced"
    )

    for j in range(8):
        if loading[
            j*5:(j+1)*5, j
        ].mean() < 0:
            loading[:, j] *= -1

    baseline = 6.7 + rng.normal(0, 0.32, 48)
    baseline += 2.0

    offaxis_loading = rng.normal(size=48)
    offaxis_loading /= np.linalg.norm(
        offaxis_loading
    )

    return loading, baseline, offaxis_loading


def psi_material(
    a, t, loading, baseline, amplitude
):
    a, t = np.broadcast_arrays(
        np.asarray(a, float),
        np.asarray(t, float)
    )

    A1 = amplitude*a**3
    A2 = amplitude*SECOND_PAIR_WEIGHT*a**4

    molecular = np.stack([
        A1*np.cos(OMEGA1*t),
        A1*np.sin(OMEGA1*t),
        A2*np.cos(OMEGA2*t),
        A2*np.sin(OMEGA2*t),
    ], axis=-1)

    spatial = spatial_basis(a)*AMP[:4]

    return (
        baseline
        + np.concatenate(
            [spatial, molecular],
            axis=-1
        ) @ loading.T
    )


amplitude = calibration_amplitude()
loading, baseline, _ = expression_parameters(SEED)

# Exact positivity correction from frozen generator.
cal_a = np.linspace(0, 1, 501)
cal_t = np.linspace(0, 1, 33)

cal_A, cal_T = np.meshgrid(cal_a, cal_t)

cal_psi = psi_material(
    cal_A, cal_T,
    loading,
    np.zeros(48),
    amplitude
)

baseline = baseline + np.maximum(
    1.5 - (baseline + cal_psi).min(axis=(0, 1)),
    0
)

# ============================================================
# 3. VERIFY FROZEN EXPRESSION TRUTH
# ============================================================

def sha256(path):
    h = hashlib.sha256()

    with open(path, "rb") as f:
        for block in iter(
            lambda: f.read(1024*1024),
            b""
        ):
            h.update(block)

    return h.hexdigest()


if VERIFY_FROZEN:
    manifest_path = (
        DATA / "manifests/dataset_manifest.json"
    )

    if not manifest_path.is_file():
        raise FileNotFoundError(
            f"Missing manifest: {manifest_path}"
        )

    manifest = json.loads(
        manifest_path.read_text()
    )

    assert manifest["dataset_id"] == (
        "synthetic_developmental_crescent_v1"
    )

    assert int(manifest["seed"]) == SEED
    assert manifest["status"] == "frozen"
    assert manifest["config"]["regime"] == "B"

    assert float(
        manifest["config"]["parallel_strength"]
    ) == 0.0

    assert float(
        manifest["config"]["offaxis_strength"]
    ) == 0.0

    truth_rel = (
        "evaluator_only/observed_hidden/cell_truth.csv.gz"
    )

    rates_rel = (
        "evaluator_only/observed_hidden/expected_rates.csv.gz"
    )

    obs_rel = (
        "train_only/public/observed_cells.csv.gz"
    )

    for rel in (truth_rel, rates_rel, obs_rel):
        p = DATA / rel

        if not p.is_file():
            raise FileNotFoundError(p)

        assert sha256(p) == manifest["file_sha256"][rel], (
            f"SHA mismatch: {rel}"
        )

    obs = pd.read_csv(
        DATA / obs_rel,
        usecols=["cell_id", "time"]
    )

    tru = pd.read_csv(
        DATA / truth_rel,
        usecols=[
            "cell_id",
            "material_a_true",
            "u_true"
        ]
    )

    rates = pd.read_csv(
        DATA / rates_rel,
        usecols=["cell_id", *GENES]
    )

    for frame in (obs, tru, rates):
        frame["cell_id"] = frame["cell_id"].astype(str)
        assert frame.cell_id.is_unique

    check = (
        obs.merge(
            tru,
            on="cell_id",
            validate="one_to_one"
        )
        .merge(
            rates,
            on="cell_id",
            validate="one_to_one"
        )
    )

    assert (
        len(check)
        == len(obs)
        == len(tru)
        == len(rates)
        == 36425
    )

    ind = np.random.default_rng(
        SEED + 53
    ).choice(
        len(check),
        size=600,
        replace=False
    )

    sample = check.iloc[ind]

    expected = (
        0.25
        * psi_material(
            sample["material_a_true"].to_numpy(float),
            sample["time"].to_numpy(float),
            loading,
            baseline,
            amplitude
        )**2
    )

    gene_indices = [
        int(g[1:]) - 1 for g in GENES
    ]

    saved = sample[
        list(GENES)
    ].to_numpy(float)

    err = np.abs(
        expected[:, gene_indices] - saved
    )

    relerr = float(np.max(
        err / np.maximum(1.0, saved)
    ))

    assert relerr < 2e-7, (
        f"Expression oracle mismatch: {relerr:.4g}"
    )

    print(
        "PASS: frozen expression verification; "
        f"max relative error={relerr:.2g}"
    )

else:
    print(
        "Offline preview: frozen truth verification disabled."
    )

# ============================================================
# 4. COMPUTE E(z,t) IN EULERIAN COORDINATES
# ============================================================

z_grid = np.linspace(0.0, 1.0, NZ)
t_grid = np.linspace(0.0, 1.0, NT)
a_grid = np.linspace(0.0, 1.0, 4001)

A_of_ZT = np.empty(
    (NZ, NT), dtype=float
)

f = a_grid.copy()
dt = 1.0 / (NT - 1)

for k, t in enumerate(t_grid):

    A_of_ZT[:, k] = np.interp(
        z_grid, f, a_grid
    )

    if k + 1 == NT:
        break

    k1 = v_z(f, t)
    k2 = v_z(f + 0.5*dt*k1, t + 0.5*dt)
    k3 = v_z(f + 0.5*dt*k2, t + 0.5*dt)
    k4 = v_z(f + dt*k3, t + dt)

    f = (
        f + dt/6.0 * (
            k1 + 2*k2 + 2*k3 + k4
        )
    )

assert np.isfinite(A_of_ZT).all()

assert np.max(
    np.abs(A_of_ZT[:, 0] - z_grid)
) < 1e-12

assert np.all(
    np.diff(A_of_ZT, axis=0) >= -1e-9
)

T_T = np.broadcast_to(
    t_grid[None, :],
    A_of_ZT.shape
)

full_rate = (
    0.25
    * psi_material(
        A_of_ZT,
        T_T,
        loading,
        baseline,
        amplitude
    )**2
)

pick = [
    int(g[1:]) - 1 for g in GENES
]

E = full_rate[:, :, pick]

assert E.shape == (NZ, NT, len(GENES))
assert np.isfinite(E).all()
assert np.all(E >= 0)

print("\nExpected expression ranges:")

for i, gene in enumerate(GENES):
    print(
        f"Gene Type {i+1} ({gene}): "
        f"{E[:,:,i].min():.2f} to "
        f"{E[:,:,i].max():.2f}"
    )

# ============================================================
# 5. PUBLICATION FONT AND STYLE
# ============================================================

available_fonts = {
    f.name for f in fm.fontManager.ttflist
}

font_candidates = [
    "Arial",
    "Liberation Sans",
    "Arimo",
    "Nimbus Sans",
    "DejaVu Sans"
]

FONT = next(
    (
        name for name in font_candidates
        if name in available_fonts
    ),
    "DejaVu Sans"
)

print("\nPublication font:", FONT)

mpl.rcParams.update({
    "font.family": FONT,
    "font.size": AXIS_TICK_SIZE,
    "axes.labelsize": AXIS_LABEL_SIZE,
    "axes.titlesize": TITLE_SIZE,
    "xtick.labelsize": AXIS_TICK_SIZE,
    "ytick.labelsize": AXIS_TICK_SIZE,
    "axes.linewidth": 0.75,
    "axes.edgecolor": INK,
    "axes.labelcolor": INK,
    "xtick.color": INK,
    "ytick.color": INK,
    "text.color": INK,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
    "svg.fonttype": "path",
    "savefig.bbox": None,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
})

# ============================================================
# 6. COMPACT 4 x 2 LAYOUT
# ============================================================

fig = plt.figure(
    figsize=(FIG_W, FIG_H),
    facecolor="white"
)

# All positions are in PHYSICAL INCHES.
#
# Previous version:
#   heatmap width  = 0.99"
#   heatmap height = 0.86"
#   bottom row y   = 0.87"
#
# New version:
#   heatmap width  = 0.92"
#   heatmap height = 1.00"
#   bottom row y   = 0.50"
#
# Net heatmap area increases by approximately 8%.

AX_W = 0.92
AX_H = 1.00

# Move left column right to clear the y-axis label.
AX_LEFTS = [0.52, 1.84]

# Top-to-bottom row positions.
ROW_BOTTOMS = [
    4.10,
    2.90,
    1.70,
    0.50
]

# Individual mini-colorbars.
CB_OFFSET = 0.93
CB_W = 0.055

# Vertical spacing between heatmaps is 0.20".
# Titles are centered within that gap.
TITLE_OFFSET = 0.10

axes = []
colorbar_axes = []

# ============================================================
# 7. EIGHT GENE EXPRESSION HEATMAPS
# ============================================================

for i, gene in enumerate(GENES):

    row, col = divmod(i, 2)

    left = AX_LEFTS[col]
    bottom = ROW_BOTTOMS[row]

    ax = fig.add_axes([
        left / FIG_W,
        bottom / FIG_H,
        AX_W / FIG_W,
        AX_H / FIG_H
    ])

    gene_min = float(E[:, :, i].min())
    gene_max = float(E[:, :, i].max())

    img = ax.pcolormesh(
        t_grid,
        z_grid,
        E[:, :, i],
        cmap="viridis",
        vmin=gene_min,
        vmax=gene_max,
        shading="auto",
        rasterized=True
    )

    # Original white expression contours.
    levels = np.linspace(
        gene_min,
        gene_max,
        7
    )[1:-1]

    ax.contour(
        t_grid,
        z_grid,
        E[:, :, i],
        levels=levels,
        colors="white",
        linewidths=0.52,
        alpha=0.84
    )

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)

    ax.set_xticks([0, 0.5, 1])
    ax.set_yticks([0, 0.5, 1])

    ax.xaxis.set_major_formatter(
        FormatStrFormatter("%g")
    )

    ax.yaxis.set_major_formatter(
        FormatStrFormatter("%g")
    )

    ax.tick_params(
        length=2.6,
        width=0.7,
        pad=1.5,
        labelsize=AXIS_TICK_SIZE
    )

    # Only bottom-row x ticks and left-column y ticks.
    ax.tick_params(
        axis="x",
        bottom=(row == 3),
        labelbottom=(row == 3)
    )

    ax.tick_params(
        axis="y",
        left=(col == 0),
        labelleft=(col == 0)
    )

    ax.spines[["top", "right"]].set_visible(False)

    # Gene Type N — centered above the heatmap.
    fig.text(
        (left + AX_W/2) / FIG_W,
        (bottom + AX_H + TITLE_OFFSET) / FIG_H,
        f"Gene Type {i+1}",
        ha="center",
        va="center",
        fontsize=TITLE_SIZE,
        fontweight="bold",
        color=INK
    )

    # --------------------------------------------------------
    # Individual colorbar with absolute count values
    # --------------------------------------------------------

    cb_left = left + CB_OFFSET

    cax = fig.add_axes([
        cb_left / FIG_W,
        bottom / FIG_H,
        CB_W / FIG_W,
        AX_H / FIG_H
    ])

    cb = fig.colorbar(
        img,
        cax=cax,
        orientation="vertical"
    )

    cb.set_ticks([
        gene_min,
        gene_max
    ])

    cb.set_ticklabels([
        f"{gene_min:.1f}",
        f"{gene_max:.1f}"
    ])

    cb.ax.tick_params(
        labelsize=CB_TICK_SIZE,
        length=1.8,
        width=0.6,
        pad=1.0
    )

    cb.outline.set_linewidth(0.6)

    axes.append(ax)
    colorbar_axes.append(cax)

# ============================================================
# 8. SHARED AXIS LABELS
# ============================================================

# Shifted left and separated from first-column tick numbers.
shared_ylabel = fig.text(
    0.10 / FIG_W,
    2.81 / FIG_H,
    "Common axis",
    rotation=90,
    ha="center",
    va="center",
    fontsize=AXIS_LABEL_SIZE,
    fontweight="normal",
    color=INK
)

# Raised relative to bottom margin but closer to the data.
shared_xlabel = fig.text(
    1.60 / FIG_W,
    0.115 / FIG_H,
    "Developmental time",
    ha="center",
    va="center",
    fontsize=AXIS_LABEL_SIZE,
    fontweight="normal",
    color=INK
)

# ============================================================
# 9. VERIFY EXPORT BOUNDS AND LABEL SPACING
# ============================================================

fig.canvas.draw()
renderer = fig.canvas.get_renderer()

bounds = fig.get_tightbbox(renderer)

print("\nCanvas dimensions:")
print(f"{FIG_W:.2f} x {FIG_H:.2f} inches")

print("\nRendered content bounds (inches):")
print([
    round(float(v), 3)
    for v in bounds.extents
])

tol = 0.005

assert bounds.x0 >= -tol, (
    "Content exceeds left boundary."
)

assert bounds.y0 >= -tol, (
    "Content exceeds bottom boundary."
)

assert bounds.x1 <= FIG_W + tol, (
    "Content exceeds right boundary."
)

assert bounds.y1 <= FIG_H + tol, (
    "Content exceeds top boundary."
)

# Explicitly verify that the shared y-axis label
# does not overlap the left-column y tick labels.
ylabel_bbox = shared_ylabel.get_window_extent(renderer)

left_tick_bboxes = [
    tick.get_window_extent(renderer)
    for ax in axes[::2]
    for tick in ax.get_yticklabels()
    if tick.get_visible()
]

min_tick_left = min(
    bbox.x0 for bbox in left_tick_bboxes
)

gap_px = min_tick_left - ylabel_bbox.x1

print(
    "Minimum shared-y-label / tick separation:",
    round(gap_px / fig.dpi, 3),
    "inches"
)

assert gap_px > 0, (
    "Common axis label overlaps y tick labels."
)

# ============================================================
# 10. EXPORT
# ============================================================

for ext in ("svg", "pdf", "png"):

    path = OUT / f"{STEM}.{ext}"

    fig.savefig(
        path,
        dpi=DPI,
        bbox_inches=None,
        facecolor="white"
    )

    print("SAVED:", path)

svg_path = OUT / f"{STEM}.svg"
root = ET.parse(svg_path).getroot()

print("\nSVG width:", root.attrib.get("width"))
print("SVG height:", root.attrib.get("height"))
print("SVG viewBox:", root.attrib.get("viewBox"))

# ============================================================
# 11. SAVE GROUND-TRUTH DATA AND PROVENANCE
# ============================================================

np.savez_compressed(
    OUT / f"{STEM}_oracle_grid.npz",
    genes=np.array(GENES),
    z=z_grid,
    t=t_grid,
    expected_counts=E.astype(np.float32),
    material_coordinate=A_of_ZT.astype(np.float32)
)

receipt = {
    "dataset_id": "synthetic_developmental_crescent_v1",
    "seed": SEED,
    "genes": list(GENES),
    "display_labels": [
        f"Gene Type {i+1}"
        for i in range(len(GENES))
    ],
    "expected_expression": (
        "E_g(z,t) = Psi_g(F_t^{-1}(z),t)^2/4"
    ),
    "model_predictions_used": False,
    "frozen_oracle_verified": bool(VERIFY_FROZEN),
    "gene_specific_absolute_count_scales": {
        gene: [
            float(E[:, :, i].min()),
            float(E[:, :, i].max())
        ]
        for i, gene in enumerate(GENES)
    },
    "figure_inches": [FIG_W, FIG_H],
    "layout": "4 rows x 2 columns",
    "heatmap_inches": [AX_W, AX_H],
    "font": FONT,
    "axes": {
        "x": "developmental_time",
        "y": "current_common_coordinate"
    }
}

(OUT / f"{STEM}_receipt.json").write_text(
    json.dumps(receipt, indent=2) + "\n"
)

print("\nPanel D complete.")

plt.show()
plt.close(fig)